# KCYPS 2018 공격성 궤적 예측 — v2 (wave 1–7, 4집단 GBTM + 반복 nested CV Random Forest + SHAP)

**Google Colab에서 위에서부터 순서대로 실행하세요.** R 없이 Python만 사용합니다.
첫 코드 셀을 실행하면 파일 업로드 창이 뜹니다 → `w1.csv` ~ `w7.csv` 선택
(부모 SES 변수를 포함하려면 `addw1.csv`도 함께 업로드).

### v2 분석 구조 (논문 프레임)
- **궤적**: G1 낮은 출발-증가형 · G2 낮은 수준 완만한 감소형 · G3 중간 수준 완만한 감소형 · G4 높은 수준 상승 후 감소형 (모양으로 자동 식별)
- **주 분석**: G4(고위험) vs 나머지, G4 vs G3(높게 출발한 아동 중 유지 vs 감소)
- **G1 분석**: G1 vs G2(낮게 출발한 아동 중 증가 vs 유지), 보조로 G1 vs G3. G1이 전체 평균 수준으로 올라오는 '규범적 증가'인지 7-4에서 확인
- **검증**: 반복 nested CV (바깥 5-fold × 3회, 안쪽 3-fold 튜닝), 모형 A(기저 공격성 제외) vs B(포함)
- **민감도**: 집단 분류 불확실성 — 사후확률 가중, 사후확률 ≥ .70 제한
- **결과**: 모든 그림(PNG 300 dpi)·표(CSV + `All_tables.xlsx`)가 `KCYPS_results.zip`으로 저장·다운로드

### v1 노트북 대비 데이터 처리 수정 (유지)
| 단계 | 기존 | 수정 |
|---|---|---|
| 공격성 점수 | `sum()`이 무응답(패널 이탈)을 **0점**으로 계산 (wave 2–6에서 170–365명) | 범위 밖 = 결측, 5/6문항 이상 응답 시 평균×6, 그 외 결측 |
| 기간 | wave 1–6 | **wave 1–7** |
| 궤적 분석 | 개인 내 표준화 + K-means (수준 정보 제거 → 평균으로의 회귀) | **GBTM (censored normal)** — 원점수의 수준과 형태를 함께 모형화, 결측 wave는 FIML |
| 궤적 라벨 결측 | 예측변수로 RF 대치 후 같은 변수로 예측 (순환) | 대치 불필요 (GBTM이 전원 분류) |
| 자아존중감 | 역문항 5·6·8·9 (2번 누락) → α = .65 | 역문항 2·5·6·8·9 → α ≈ .83 |
| 그릿 | 4점 척도를 5점으로 역코딩 | 5 − x |
| 범위 밖 값 | 교사관계 14번(0, 5–15), 스마트폰 의존 15번(5) 그대로 합산 | 결측 처리 |
| 스마트폰 미사용자 | 의존도 0점 (198명) | 결측 + '미사용' 지시변수 |
| 변수선택 | 학습셋 전체에서 선택 후 CV | Pipeline 안에서 선택 (CV 누수 방지) |
| 기저 공격성 | 미포함 | 모형 A(제외) vs B(포함) 비교, ΔAUC |
| SHAP | 기준값 0으로 표기, 사례 수동 선택 | 기준값 = 평균 예측확률, 시드 고정 무작위 사례, BH-FDR |

실행 시간(Colab 무료, CPU 2코어 기준): GBTM 약 5분, 예측모형 약 40–60분, 민감도 분석 약 10–15분.  
먼저 코드가 도는지만 보려면 8절의 `FAST_MODE = True` 로 바꿔 실행하세요 (약 15분).


In [ ]:
%pip install -q shap

## 0. 환경 설정 · 파일 업로드

In [ ]:
# ============================================================
# 0. 환경 설정
#    - Colab: 아래 셀 실행 → 파일 업로드 창이 뜨면 w1.csv ~ w7.csv 선택
#      (부모 SES 변수를 쓰려면 addw1.csv 도 함께 업로드)
#    - Google Drive 사용 시 USE_DRIVE = True, DRIVE_DIR 수정
# ============================================================
import os, sys, glob, json, shutil, warnings, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 60, "display.max_rows", 200)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = False                                   # Drive에 파일이 있으면 True
DRIVE_DIR = "/content/drive/MyDrive/KCYPS2018"      # Drive 경로 (USE_DRIVE=True 일 때)

if IN_COLAB:
    DATA_DIR = "/content/data"
    OUT_DIR = "/content/outputs"
else:
    DATA_DIR = os.environ.get("KCYPS_DATA_DIR", "./data")
    OUT_DIR = os.environ.get("KCYPS_OUT_DIR", "./outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

WAVES = list(range(1, 8))                           # wave 1–7 (2018–2024)
REQUIRED = [f"w{w}.csv" for w in WAVES]

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for f in REQUIRED + ["addw1.csv"]:
        src = os.path.join(DRIVE_DIR, f)
        if os.path.exists(src):
            shutil.copy(src, DATA_DIR)

missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing and IN_COLAB:
    from google.colab import files
    print("업로드할 파일:", missing, "(+ 선택: addw1.csv)")
    uploaded = files.upload()
    for name in uploaded:
        # 업로드 시 이름이 'w1 (1).csv' 처럼 바뀌는 경우 정리
        clean = name.split(" ")[0] if name.split(" ")[0].endswith(".csv") else name
        shutil.move(name, os.path.join(DATA_DIR, clean))
    missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    raise FileNotFoundError(f"다음 파일이 {DATA_DIR} 에 없습니다: {missing}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Python", sys.version.split()[0], "| DATA_DIR =", os.path.abspath(DATA_DIR), "| OUT_DIR =", os.path.abspath(OUT_DIR))
print("파일 확인 완료:", sorted(os.listdir(DATA_DIR)))


## 1. 원자료 읽기

In [ ]:
# ============================================================
# 1. 원자료 읽기
#    - wave마다 인코딩이 다름 (w1–w6: latin1, w7: cp949)
#    - w7은 무응답이 공백 문자열(' ')로 저장되어 있어 숫자 변환 시 NaN 처리
# ============================================================
def read_kcyps(path):
    last_err = None
    for enc in ["utf-8-sig", "cp949", "euc-kr", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False), enc
        except (UnicodeDecodeError, UnicodeError) as e:
            last_err = e
    raise last_err

def num(s):
    """문자열/공백 → NaN, 숫자형 변환"""
    return pd.to_numeric(s, errors="coerce")

raw = {}
for w in WAVES:
    df_w, enc = read_kcyps(os.path.join(DATA_DIR, f"w{w}.csv"))
    df_w["ID"] = num(df_w["ID"]).astype("Int64")
    raw[w] = df_w
    print(f"w{w}: {df_w.shape[0]:>5} rows × {df_w.shape[1]:>3} cols  (encoding={enc})")

base_ids = raw[1]["ID"].dropna().astype(int)
print("\n기저(wave 1) 패널 N =", base_ids.nunique())


## 2. 공격성 점수 (wave 1–7) — 0점 버그 수정
`old_code_zero_scores` 열은 기존 코드에서 0점으로 잘못 계산되던 사례 수입니다.

In [ ]:
# ============================================================
# 2. 공격성 점수 (wave 1–7)
#    [기존 코드의 버그 수정]
#    기존: df[items].sum(axis=1) → 6문항이 모두 결측(패널 이탈)이어도 0점
#    수정: 범위(1–4) 밖 값 = 결측, 6문항 중 5문항 이상 응답 시 평균×6 (prorating),
#          그 외에는 결측(NaN)으로 둔다 → GBTM이 결측 wave를 그대로 처리
# ============================================================
def scale_score(df, items, suffix, lo, hi, reverse=(), min_valid=None, prorate=True):
    cols = [f"{it}{suffix}" for it in items]
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"없는 변수: {missing_cols}")
    X = df[cols].apply(num)
    X = X.where((X >= lo) & (X <= hi))                      # 범위 밖 = 결측
    for it in reverse:
        X[f"{it}{suffix}"] = lo + hi - X[f"{it}{suffix}"]   # 역코딩
    k = len(cols)
    if min_valid is None:
        min_valid = int(np.ceil(0.8 * k))                   # 기본: 80% 이상 응답
    n_valid = X.notna().sum(axis=1)
    score = X.mean(axis=1) * k if prorate else X.sum(axis=1, min_count=k)
    return score.where(n_valid >= min_valid), X

AGG_ITEMS = [f"YPSY4B{i:02d}" for i in range(1, 7)]
AGG_MIN, AGG_MAX = 6, 24
agg_cols = [f"Agg_w{w}" for w in WAVES]

agg = pd.DataFrame({"ID": base_ids.values})
diag = []
for w in WAVES:
    d = raw[w]
    s, X = scale_score(d, AGG_ITEMS, f"w{w}", 1, 4, min_valid=5)
    tmp = pd.DataFrame({"ID": d["ID"].astype("Int64"), f"Agg_w{w}": s.values})
    agg = agg.merge(tmp, on="ID", how="left")
    buggy = d[[f"{it}w{w}" for it in AGG_ITEMS]].apply(num).sum(axis=1)   # 기존 방식
    diag.append({
        "wave": w, "year": 2017 + w, "rows_in_file": len(d),
        "valid_score_n": int(s.notna().sum()),
        "retention_%": round(100 * s.notna().sum() / len(base_ids), 1),
        "min": s.min(), "max": s.max(), "mean": round(s.mean(), 2), "sd": round(s.std(), 2),
        "at_floor_6_%": round(100 * (s == AGG_MIN).mean() / s.notna().mean(), 1),
        "old_code_zero_scores": int((buggy == 0).sum()),
        "alpha": None,
    })
    # Cronbach's alpha (complete cases)
    Xc = X.dropna()
    k = Xc.shape[1]
    diag[-1]["alpha"] = round(k / (k - 1) * (1 - Xc.var().sum() / Xc.sum(axis=1).var()), 3)

agg_diag = pd.DataFrame(diag)
print("▶ 공격성 점수 진단 (old_code_zero_scores = 기존 코드에서 0점으로 잘못 계산된 사례 수)")
display(agg_diag) if "display" in globals() else print(agg_diag)

assert agg[agg_cols].min().min() >= AGG_MIN, "최솟값이 6보다 작습니다 → 점수 계산 확인 필요"
agg["n_waves_observed"] = agg[agg_cols].notna().sum(axis=1)
print("\n관측 wave 수 분포:\n", agg["n_waves_observed"].value_counts().sort_index().to_string())
agg_diag.to_csv(os.path.join(OUT_DIR, "diag_aggression_by_wave.csv"), index=False, encoding="utf-8-sig")


## 3. 기저(wave 1) 예측변수
역코딩 점검표에서 item-rest r이 음수인 문항은 코드북으로 역문항 여부를 다시 확인하세요.

In [ ]:
# ============================================================
# 3. 기저(wave 1) 예측변수
#    [기존 코드 대비 수정 사항]
#    - 범위 밖 값(예: 교사관계 14번 문항의 0, 5–15; 스마트폰 의존 15번 문항의 5) → 결측
#    - 문항 무응답을 0점으로 더하지 않음 (80% 이상 응답 시 평균×문항수)
#    - 자아존중감 역문항: 2, 5, 6, 8, 9 (원척도 기준; 기존 코드는 2번 누락·8번만 포함 → α=.65)
#    - 그릿: 4점 척도이므로 역코딩은 5 - x (기존 코드는 6 - x)
#    - 스마트폰 미사용자: 의존도/사용빈도 = 결측 + '스마트폰 미사용' 지시변수
#    - 양육: 긍정(따스함·자율성지지·구조) / 부정(거부·강요·비일관성) 합성
#    ※ 역문항 목록은 아래 '역코딩 점검'에서 데이터로 재확인한다 (코드북으로도 최종 확인 권장)
# ============================================================
SUF = "w1"
d1 = raw[1].copy()

SCALES = {
    # 변수명: (라벨, 문항 리스트, 최소, 최대, 역문항)
    "somatic":      ("Somatic symptoms",            [f"YPSY4C{i:02d}" for i in range(1, 9)],  1, 4, []),
    "attention":    ("Attention problems",          [f"YPSY4A{i:02d}" for i in range(1, 8)],  1, 4, []),
    "depressive":   ("Depressive symptoms",         [f"YPSY4E{i:02d}" for i in range(1, 11)], 1, 4, []),
    "withdrawal":   ("Social withdrawal",           [f"YPSY4D{i:02d}" for i in range(1, 6)],  1, 4, []),
    "self_esteem":  ("Self-esteem",                 [f"YPSY3A{i:02d}" for i in range(1, 11)], 1, 4,
                     ["YPSY3A02", "YPSY3A05", "YPSY3A06", "YPSY3A08", "YPSY3A09"]),
    "happiness":    ("Happiness",                   [f"YPSY2A{i:02d}" for i in range(1, 5)],  1, 4, ["YPSY2A04"]),
    "life_sat":     ("Life satisfaction",           [f"YPSY1A{i:02d}" for i in range(1, 6)],  1, 4, []),
    "grit":         ("Grit",                        [f"YPSY7A{i:02d}" for i in range(1, 9)],  1, 4,
                     ["YPSY7A01", "YPSY7A03", "YPSY7A05", "YPSY7A06"]),
    "coop_group":   ("Cooperation (group facilitation)", [f"YPSY5A{i:02d}" for i in range(1, 7)],   1, 4, []),
    "coop_follow":  ("Cooperation (followership)",  [f"YPSY5A{i:02d}" for i in range(7, 10)],  1, 4, []),
    "coop_conflict":("Cooperation (conflict resolution)", [f"YPSY5A{i:02d}" for i in range(10, 15)], 1, 4, []),
    "eng_dedication":("Academic engagement (dedication)", [f"YINT2A{i:02d}" for i in range(1, 5)],  1, 4, []),
    "eng_vigor":    ("Academic engagement (vigor)",  [f"YINT2A{i:02d}" for i in range(5, 9)],  1, 4, []),
    "eng_efficacy": ("Academic engagement (self-efficacy)", [f"YINT2A{i:02d}" for i in range(9, 13)], 1, 4, []),
    "eng_absorption":("Academic engagement (absorption)", [f"YINT2A{i:02d}" for i in range(13, 17)], 1, 4, []),
    "sleep_quality":("Sleep quality",               ["YTIM1C01", "YTIM1C02"],                1, 4, []),
    "smart_dep":    ("Smartphone dependency",       [f"YMDA1C{i:02d}" for i in range(1, 16)], 1, 4,
                     ["YMDA1C05", "YMDA1C10", "YMDA1C15"]),
    "smart_use":    ("Smartphone use frequency",    [f"YMDA1B{i:02d}" for i in range(1, 12)], 1, 4, []),
    "peer":         ("Peer relationship",           [f"YEDU2A{i:02d}" for i in range(1, 14)], 1, 4,
                     [f"YEDU2A{i:02d}" for i in range(9, 14)]),
    "teacher":      ("Teacher–child relationship",  [f"YEDU3A{i:02d}" for i in range(1, 15)], 1, 4, []),
    "parenting_pos":("Warm, autonomy-supportive, and structured parenting",
                     [f"YFAM2{s}{i:02d}" for s in "ACE" for i in range(1, 5)], 1, 4, []),
    "parenting_neg":("Rejecting, coercive, and inconsistent parenting",
                     [f"YFAM2{s}{i:02d}" for s in "BDF" for i in range(1, 5)], 1, 4, []),
    "parent_talk":  ("Parent conversation time",    ["YTIM1D01", "YTIM1D02"],                1, 7, []),
}
# 양육 하위척도(보충자료용 α 보고)
PARENT_SUB = {"warmth": "A", "rejection": "B", "autonomy_support": "C",
              "coercion": "D", "structure": "E", "inconsistency": "F"}

def cronbach_alpha(X):
    Xc = X.dropna()
    k = Xc.shape[1]
    if k < 2 or len(Xc) < 10:
        return np.nan
    return k / (k - 1) * (1 - Xc.var(ddof=1).sum() / Xc.sum(axis=1).var(ddof=1))

pred = pd.DataFrame({"ID": d1["ID"].astype("Int64")})
alpha_rows, keying_rows = [], []
for var, (label, items, lo, hi, rev) in SCALES.items():
    s, X = scale_score(d1, items, SUF, lo, hi, reverse=rev)
    pred[var] = s.values
    alpha_rows.append({"variable": var, "label": label, "n_items": len(items),
                       "range": f"{lo}-{hi}", "reverse_items": ",".join(r[-2:] for r in rev),
                       "alpha_w1": round(cronbach_alpha(X), 3), "n_valid": int(s.notna().sum())})
    # 역코딩 점검: 역코딩 후 '문항-나머지' 상관이 음수면 경고
    tot = X.sum(axis=1, min_count=1)
    for c in X.columns:
        r = X[c].corr(tot - X[c].fillna(0))
        keying_rows.append({"variable": var, "item": c.replace(SUF, ""), "item_rest_r": round(r, 2)})

for sub, code in PARENT_SUB.items():
    _, X = scale_score(d1, [f"YFAM2{code}{i:02d}" for i in range(1, 5)], SUF, 1, 4)
    alpha_rows.append({"variable": f"parenting_{sub}", "label": f"Parenting subscale: {sub}", "n_items": 4,
                       "range": "1-4", "reverse_items": "", "alpha_w1": round(cronbach_alpha(X), 3),
                       "n_valid": int(X.notna().all(axis=1).sum())})

alpha_df = pd.DataFrame(alpha_rows)
keying_df = pd.DataFrame(keying_rows)
bad = keying_df[keying_df["item_rest_r"] < 0.05]
print("▶ 척도별 Cronbach α (wave 1)")
print(alpha_df.to_string(index=False))
print("\n▶ 역코딩 점검 — item-rest r < .05 인 문항 (코드북으로 역문항 여부 재확인 권장):")
print(bad.to_string(index=False) if len(bad) else "  없음")

# ----- 배경 변수 -----
pred["female"] = (num(d1["YGENDERw1"]) == 2).astype(float)          # 1=남, 2=여 → 여아=1
pred["age"] = ((2018 - num(d1["YBRT1Aw1"])) + (8 - num(d1["YBRT1Bw1"])) / 12).round(2)
pred["region_size"] = num(d1["ARA2Aw1"])                             # 1=대도시, 2=중소도시, 3=읍면
pred["school_region"] = num(d1["ARA1Aw1"])                           # 17개 시도 (거주지역은 중복이라 제외)
pred["no_smartphone"] = pred["smart_dep"].isna().astype(float)

# ----- (선택) 부모 SES: addw1.csv 가 있으면 병합 -----
SES_FILE = os.path.join(DATA_DIR, "addw1.csv")
if os.path.exists(SES_FILE):
    ses, _ = read_kcyps(SES_FILE)
    ses["ID"] = num(ses["ID"]).astype("Int64")
    keep = [c for c in ["PINCOMEw1", "PSCHOOL1w1", "PSCHOOL2w1", "PJOB1w1", "PJOB2w1", "PFAM1A00w1"] if c in ses.columns]
    ses = ses[["ID"] + keep].copy()
    for c in keep:
        ses[c] = num(ses[c])
    # ※ 학력 코드 8·10(모름/기타 등 추정)은 코드북 확인 후 처리: 여기서는 결측으로 둔다
    for c in ["PSCHOOL1w1", "PSCHOOL2w1"]:
        if c in ses.columns:
            ses.loc[ses[c].isin([8, 9, 10]), c] = np.nan
    ses = ses.rename(columns={"PINCOMEw1": "income", "PSCHOOL1w1": "father_edu", "PSCHOOL2w1": "mother_edu",
                              "PJOB1w1": "father_job", "PJOB2w1": "mother_job", "PFAM1A00w1": "family_type"})
    pred = pred.merge(ses, on="ID", how="left")
    print("\n✅ addw1.csv 병합:", [c for c in ses.columns if c != "ID"])
else:
    print("\n⚠️ addw1.csv 없음 → 부모 SES 변수 없이 진행 (업로드하면 자동 포함)")

LABELS = {k: v[0] for k, v in SCALES.items()}
LABELS.update({"female": "Female sex", "age": "Age", "region_size": "School location region size",
               "school_region": "School region (province)", "no_smartphone": "No smartphone",
               "income": "Household income", "father_edu": "Father educational attainment",
               "mother_edu": "Mother educational attainment", "father_job": "Father occupation",
               "mother_job": "Mother occupation", "family_type": "Family type",
               "Agg_w1": "Baseline aggression (wave 1)"})

alpha_df.to_csv(os.path.join(OUT_DIR, "TableS1_alpha_w1.csv"), index=False, encoding="utf-8-sig")
keying_df.to_csv(os.path.join(OUT_DIR, "check_item_keying_w1.csv"), index=False, encoding="utf-8-sig")
print("\n예측변수 결측 수:\n", pred.drop(columns="ID").isna().sum()[lambda s: s > 0].to_string())


## 4. GBTM / LCGA 구현 (Python)
Nagin(2005)의 집단기반 궤적모형. 공격성 점수의 바닥효과(6점에 10–16%)를 고려해 censored normal 분포를 사용합니다. `CENSORED = False`로 바꾸면 정규분포 LCGA.

In [ ]:
# ============================================================
# 4. GBTM / LCGA (Python 구현, R 불필요)
#    - Nagin의 집단기반 궤적모형(GBTM)과 같은 모형:
#      각 집단 k의 평균궤적 μ_k(t) = β_k0 + β_k1·t + β_k2·t² (+ β_k3·t³)
#      집단 내 개인차 없음(= LCGA), 공통 σ, 집단 비율은 다항로짓
#    - 분포: censored normal (CNORM; Stata traj 기본값)
#      공격성 점수는 6점(최솟값)에 10–16%가 몰려 있어(바닥효과) 6점을 좌측 검열로 처리
#      censored=False 로 두면 일반 정규 LCGA
#    - 결측 wave는 우도에서 제외 (FIML, MAR 가정) → 일부 wave만 응답한 아동도 분석에 포함
#    - 추정: 해석적 기울기 + L-BFGS-B, 여러 시작값 중 최대 우도 선택
# ============================================================
from scipy.optimize import minimize
from scipy.special import log_ndtr, logsumexp
from scipy.stats import norm

class GBTM:
    def __init__(self, n_groups, orders=3, censored=True, lo=6.0, hi=24.0,
                 n_starts=10, random_state=42, maxiter=2000):
        self.K = n_groups
        self.orders = [orders] * n_groups if np.isscalar(orders) else list(orders)
        assert len(self.orders) == self.K
        self.censored, self.lo, self.hi = censored, lo, hi
        self.n_starts, self.rs, self.maxiter = n_starts, random_state, maxiter

    # ---------- 설계행렬 ----------
    def _design(self, times):
        t = (np.asarray(times, float) - self.t_center) / self.t_scale
        pmax = max(self.orders)
        return np.vander(t, pmax + 1, increasing=True)          # (T, pmax+1)

    def _unpack(self, par):
        betas, i = [], 0
        for p in self.orders:
            betas.append(par[i:i + p + 1]); i += p + 1
        log_sigma = par[i]; i += 1
        theta = np.concatenate([[0.0], par[i:i + self.K - 1]])
        return betas, log_sigma, theta

    # ---------- 우도 + 기울기 ----------
    def _loglik_grad(self, par, need_grad=True):
        Y, M, X = self.Y_, self.M_, self.X_
        betas, log_sigma, theta = self._unpack(par)
        sigma = np.exp(log_sigma)
        logpi = theta - logsumexp(theta)
        N, T = Y.shape
        ll_k = np.zeros((N, self.K))
        gmu, gs = [], []
        F = M & (Y <= self.lo) if self.censored else np.zeros_like(M)
        C = M & (Y >= self.hi) if self.censored else np.zeros_like(M)
        I = M & ~F & ~C
        Y0 = np.where(M, Y, 0.0)
        for k, b in enumerate(betas):
            mu = X[:, :len(b)] @ b                               # (T,)
            z = (Y0 - mu) / sigma
            ll = np.zeros_like(Y0)
            ll[I] = norm.logpdf(z[I]) - log_sigma
            if self.censored:
                ll[F] = log_ndtr(z[F])
                ll[C] = log_ndtr(-z[C])
            ll_k[:, k] = ll.sum(axis=1)
            if need_grad:
                g_mu = np.zeros_like(Y0); g_s = np.zeros_like(Y0)
                g_mu[I] = z[I] / sigma
                g_s[I] = -1.0 + z[I] ** 2
                if self.censored:
                    lamF = np.exp(norm.logpdf(z[F]) - log_ndtr(z[F]))
                    lamC = np.exp(norm.logpdf(z[C]) - log_ndtr(-z[C]))
                    g_mu[F] = -lamF / sigma;  g_s[F] = -z[F] * lamF
                    g_mu[C] = lamC / sigma;   g_s[C] = z[C] * lamC
                gmu.append(g_mu); gs.append(g_s)
        joint = ll_k + logpi
        li = logsumexp(joint, axis=1)
        LL = li.sum()
        if not need_grad:
            return LL, None, joint - li[:, None]
        W = np.exp(joint - li[:, None])                           # 사후확률 (N,K)
        grads = []
        for k, b in enumerate(betas):
            grads.append((W[:, [k]] * gmu[k]).sum(axis=0) @ X[:, :len(b)])
        g_logs = sum((W[:, [k]] * gs[k]).sum() for k in range(self.K))
        pi = np.exp(logpi)
        g_theta = (W - pi).sum(axis=0)[1:]
        grad = np.concatenate(grads + [[g_logs], g_theta])
        return LL, grad, np.log(np.clip(W, 1e-300, None))

    # ---------- 시작값 ----------
    def _start_values(self, rng, labels):
        X = self.X_
        par = []
        Ymean = np.nanmean(np.where(self.M_, self.Y_, np.nan), axis=0)
        for k, p in enumerate(self.orders):
            sel = labels == k
            if sel.sum() < 5:
                yk = Ymean + rng.normal(0, 1, len(Ymean))
            else:
                yk = np.nanmean(np.where(self.M_[sel], self.Y_[sel], np.nan), axis=0)
                yk = np.where(np.isnan(yk), Ymean, yk)
            b, *_ = np.linalg.lstsq(X[:, :p + 1], yk, rcond=None)
            par.extend(b)
        resid_sd = np.nanstd(np.where(self.M_, self.Y_, np.nan))
        par.append(np.log(resid_sd * 0.8))
        counts = np.bincount(labels, minlength=self.K) + 1.0
        par.extend(np.log(counts[1:] / counts[0]))
        return np.array(par, float)

    def fit(self, Y, times, verbose=False):
        from sklearn.cluster import KMeans
        self.Y_ = np.asarray(Y, float)
        self.M_ = ~np.isnan(self.Y_)
        self.times_ = np.asarray(times, float)
        self.t_center, self.t_scale = self.times_.mean(), (self.times_.max() - self.times_.min()) / 2
        self.X_ = self._design(self.times_)
        N = len(self.Y_)
        rng = np.random.default_rng(self.rs)
        # 개인 평균으로 결측 채운 자료 → k-means 기반 시작값 + 무작위 시작값
        Yfill = np.where(self.M_, self.Y_, np.nanmean(np.where(self.M_, self.Y_, np.nan), axis=1, keepdims=True))
        results = []
        for s in range(self.n_starts):
            if s == 0:
                lab = KMeans(self.K, n_init=10, random_state=self.rs).fit_predict(Yfill) if self.K > 1 else np.zeros(N, int)
            elif s % 2 == 1 and self.K > 1:
                lab = KMeans(self.K, n_init=1, random_state=self.rs + s).fit_predict(
                    Yfill + rng.normal(0, 1.0, Yfill.shape))
            else:
                lab = rng.integers(0, self.K, N)
            x0 = self._start_values(rng, lab)
            obj = lambda p: tuple(-v for v in self._loglik_grad(p)[:2])
            try:
                res = minimize(obj, x0, jac=True, method="L-BFGS-B",
                               options={"maxiter": self.maxiter, "gtol": 1e-6})
                results.append((-res.fun, res.x, res.success))
            except (FloatingPointError, ValueError):
                continue
            if verbose:
                print(f"  start {s + 1:>2}: LL = {-res.fun:.3f}")
        results.sort(key=lambda r: -r[0])
        self.loglik_, self.params_, self.converged_ = results[0]
        self.start_lls_ = np.array([r[0] for r in results])
        self.n_replicated_ = int((np.abs(self.start_lls_ - self.loglik_) < 0.01).sum())
        _, _, logW = self._loglik_grad(self.params_, need_grad=False)
        self.post_ = np.exp(logW)
        self._order_groups()
        return self

    def _order_groups(self):
        """집단을 기저 수준(t=첫 wave 예측값) 오름차순으로 재배열"""
        betas, log_sigma, theta = self._unpack(self.params_)
        first = [self.X_[0, :len(b)] @ b for b in betas]
        order = np.argsort(first)
        self.orders = [self.orders[i] for i in order]
        betas = [betas[i] for i in order]
        theta = theta[order]; theta = theta - theta[0]
        self.params_ = np.concatenate(betas + [[log_sigma], theta[1:]])
        self.post_ = self.post_[:, order]

    # ---------- 결과 ----------
    @property
    def n_params(self):
        return sum(p + 1 for p in self.orders) + 1 + (self.K - 1)

    def group_proportions(self):
        _, _, theta = self._unpack(self.params_)
        return np.exp(theta - logsumexp(theta))

    def predict_trajectory(self, times=None):
        times = self.times_ if times is None else np.asarray(times, float)
        X = self._design(times)
        betas, _, _ = self._unpack(self.params_)
        return np.column_stack([X[:, :len(b)] @ b for b in betas])   # (T, K) 잠재 평균

    def expected_observed(self, times=None):
        """검열을 반영한 관측척도 기대값 E[y] (그래프용)"""
        mu = self.predict_trajectory(times)
        if not self.censored:
            return mu
        s = np.exp(self._unpack(self.params_)[1])
        a, b = (self.lo - mu) / s, (self.hi - mu) / s
        return (self.lo * norm.cdf(a) + self.hi * norm.sf(b)
                + mu * (norm.cdf(b) - norm.cdf(a)) + s * (norm.pdf(a) - norm.pdf(b)))

    def fit_stats(self):
        N, K, P, LL = len(self.Y_), self.K, self.n_params, self.loglik_
        W = self.post_
        modal = W.argmax(axis=1)
        pi = self.group_proportions()
        ent = 1.0 if K == 1 else 1 + (W * np.log(np.clip(W, 1e-300, None))).sum() / (N * np.log(K))
        avepp = [W[modal == k, k].mean() if (modal == k).any() else np.nan for k in range(K)]
        occ = [(a / (1 - a)) / (p / (1 - p)) if K > 1 and a < 1 else np.inf for a, p in zip(avepp, pi)]
        return {
            "groups": K, "orders": "".join(map(str, self.orders)), "n_params": P, "LL": round(LL, 2),
            "AIC": round(-2 * LL + 2 * P, 2), "BIC": round(-2 * LL + P * np.log(N), 2),
            "aBIC": round(-2 * LL + P * np.log((N + 2) / 24), 2),
            "BIC_Nagin": round(LL - 0.5 * P * np.log(N), 2),
            "entropy": round(ent, 3),
            "min_AvePP": round(np.nanmin(avepp), 3),
            "min_OCC": round(np.nanmin(occ), 1) if K > 1 else np.nan,
            "smallest_group_%": round(100 * pi.min(), 1),
            "group_%_est": " / ".join(f"{100 * p:.1f}" for p in pi),
            "group_%_modal": " / ".join(f"{100 * (modal == k).mean():.1f}" for k in range(K)),
            "AvePP": " / ".join(f"{a:.2f}" for a in avepp),
            "replicated_starts": f"{self.n_replicated_}/{len(self.start_lls_)}",
            "converged": bool(self.converged_),
        }

    def standard_errors(self, eps=1e-5):
        """해석적 기울기의 수치미분으로 관측 정보행렬 → SE"""
        p0 = self.params_.copy()
        P = len(p0)
        H = np.zeros((P, P))
        for j in range(P):
            e = np.zeros(P); e[j] = eps
            g1 = self._loglik_grad(p0 + e)[1]; g2 = self._loglik_grad(p0 - e)[1]
            H[:, j] = (g1 - g2) / (2 * eps)
        H = (H + H.T) / 2
        cov = np.linalg.pinv(-H)
        return np.sqrt(np.clip(np.diag(cov), 0, None))

    def parameter_table(self):
        se = self.standard_errors()
        rows, i = [], 0
        names = ["Intercept", "Linear", "Quadratic", "Cubic"]
        for k, p in enumerate(self.orders):
            for j in range(p + 1):
                est = self.params_[i]
                rows.append({"group": k + 1, "parameter": names[j], "estimate": est, "SE": se[i],
                             "z": est / se[i] if se[i] > 0 else np.nan,
                             "p": 2 * norm.sf(abs(est / se[i])) if se[i] > 0 else np.nan})
                i += 1
        rows.append({"group": "", "parameter": "Sigma", "estimate": np.exp(self.params_[i]),
                     "SE": np.exp(self.params_[i]) * se[i], "z": np.nan, "p": np.nan})
        return pd.DataFrame(rows)


def refine_orders(Y, times, K, start_order=3, alpha=0.05, **kw):
    """Nagin 권장 절차: 각 집단의 최고차항이 유의하지 않으면 차수를 낮춰 재추정"""
    orders = [start_order] * K
    while True:
        m = GBTM(K, orders=orders, **kw).fit(Y, times)
        tab = m.parameter_table()
        changed = False
        new = list(m.orders)
        for k in range(K):
            g = tab[tab["group"] == k + 1]
            top = g.iloc[-1]
            if new[k] > 1 and top["p"] > alpha:
                new[k] -= 1; changed = True
        if not changed:
            return m
        orders = new


## 5. 집단 수 결정
자동 제안 기준: 최소 AvePP ≥ .70, 최소 집단 ≥ 5% 를 만족하는 모형 중 BIC 최소. 해석을 함께 보고 `K_FINAL`로 확정하세요.

In [ ]:
# ============================================================
# 5. 집단 수 결정 (1–6 집단, 3차 다항식 시작)
#    선택 기준 (Nagin, 2005; van de Schoot et al., 2017 GRoLTS):
#      ① BIC 감소폭(엘보우)  ② 최소 평균사후확률(AvePP) ≥ .70
#      ③ 최소 집단 비율 ≥ 5%  ④ 시작값 반복 재현  ⑤ 해석 가능성
#    → 자동 제안 후, K_FINAL 에 직접 숫자를 넣어 덮어쓸 수 있음
# ============================================================
Y_traj = agg[agg_cols].to_numpy(float)
TIMES = np.array(WAVES, float)
MIN_WAVES = 1            # 1개 wave 이상 관측된 아동 모두 포함 (FIML). 민감도 분석 시 3 등으로 변경
use = agg["n_waves_observed"] >= MIN_WAVES
print(f"궤적 분석 표본: N = {use.sum()} (관측 wave ≥ {MIN_WAVES})")

K_RANGE = range(1, 7)
N_STARTS = 10            # 시작값 수 (최종 보고 전에는 20 이상 권장)
CENSORED = True          # True = censored normal GBTM, False = 정규 LCGA

models, fit_rows = {}, []
for K in K_RANGE:
    t0 = time.time()
    m = GBTM(K, orders=3, censored=CENSORED, n_starts=N_STARTS, random_state=RANDOM_STATE).fit(Y_traj[use], TIMES)
    models[K] = m
    st = m.fit_stats(); st["sec"] = round(time.time() - t0, 1)
    fit_rows.append(st)
    print(f"K={K}: BIC={st['BIC']:.1f}  entropy={st['entropy']}  minAvePP={st['min_AvePP']}  "
          f"smallest={st['smallest_group_%']}%  replicated={st['replicated_starts']}")
fit_table = pd.DataFrame(fit_rows)
fit_table.to_csv(os.path.join(OUT_DIR, "Table_GBTM_fit.csv"), index=False, encoding="utf-8-sig")

# ---- 자동 제안: 기준 ②③을 만족하는 모형 중 BIC 최소 ----
ok = fit_table[(fit_table["groups"] > 1) & (fit_table["min_AvePP"] >= .70) & (fit_table["smallest_group_%"] >= 5)]
K_SUGGEST = int(ok.loc[ok["BIC"].idxmin(), "groups"]) if len(ok) else 2
K_FINAL = None           # ← 직접 정하려면 숫자 입력 (예: K_FINAL = 4)
K_FINAL = K_FINAL or K_SUGGEST
print(f"\n▶ 제안 집단 수 = {K_SUGGEST}  |  최종 사용 = {K_FINAL}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(fit_table["groups"], fit_table["BIC"], "o-"); ax[0].set_xlabel("Number of groups"); ax[0].set_ylabel("BIC")
ax[1].plot(fit_table["groups"], fit_table["min_AvePP"], "o-", label="min AvePP")
ax[1].plot(fit_table["groups"], fit_table["entropy"], "s--", label="Entropy")
ax[1].axhline(.70, color="grey", lw=.8, ls=":"); ax[1].set_xlabel("Number of groups"); ax[1].legend(frameon=False)
for a in ax: a.axvline(K_FINAL, color="grey", lw=.8, ls="--")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "FigS_model_selection.png"), dpi=300); plt.show()
display(fit_table)


## 6. 최종 궤적 모형 · 집단 번호(G1–G4) · Figure 1
집단 비율은 **모형 추정 비율(Estimated %)**과 **최빈 배정 비율(Assigned %)**을 모두 보고하세요 (GRoLTS).

In [ ]:
# ============================================================
# 6. 최종 모형: 다항식 차수 정리 → 집단 번호·이름 지정 → Figure 1 / 궤적 표
#    집단 번호(논문 표기):
#      G1 = 낮은 출발-증가형      (Low-starting increasing)       ← 출발→끝 변화가 가장 큰 집단
#      G2 = 낮은 수준 완만한 감소형 (Low declining)                ← 나머지 중 평균 수준 최저
#      G3 = 중간 수준 완만한 감소형 (Moderate declining)           ← 나머지
#      G4 = 높은 수준 상승 후 감소형 (High rising-then-declining)  ← 나머지 중 평균 수준 최고
#    집단 순서는 추정마다 바뀔 수 있으므로 '모양'으로 자동 식별한다 (K = 4일 때).
# ============================================================
REFINE_ORDERS = True     # 최고차항이 유의하지 않으면 차수를 낮춤 (Nagin 절차)
N_STARTS_FINAL = 20      # 최종 모형은 시작값을 늘려 국소해 방지 (투고 전 50 권장)
if REFINE_ORDERS:
    final = refine_orders(Y_traj[use], TIMES, K_FINAL, start_order=3, censored=CENSORED,
                          n_starts=N_STARTS_FINAL, random_state=RANDOM_STATE)
else:
    final = models[K_FINAL]
print("최종 모형 다항식 차수 (추정 순서):", final.orders)
final_stats = final.fit_stats()
print(pd.Series(final_stats).to_string())

E_raw = final.expected_observed()                 # (T, K) 관측척도 기대값 (추정 순서)
first, last, level = E_raw[0], E_raw[-1], E_raw.mean(axis=0)

GROUP_LABELS = {1: "Low-starting increasing", 2: "Low declining",
                3: "Moderate declining", 4: "High rising-then-declining"}
GROUP_LABELS_KO = {1: "낮은 출발-증가형", 2: "낮은 수준 완만한 감소형",
                   3: "중간 수준 완만한 감소형", 4: "높은 수준 상승 후 감소형"}

if final.K == 4:
    k_g1 = int(np.argmax(last - first))
    rest = [k for k in range(4) if k != k_g1]
    k_g4 = max(rest, key=lambda k: level[k])
    k_g2 = min(rest, key=lambda k: level[k])
    k_g3 = [k for k in rest if k not in (k_g4, k_g2)][0]
    IDX2G = {k_g1: 1, k_g2: 2, k_g3: 3, k_g4: 4}  # 추정 순서 index → 논문 집단 번호
else:                                             # K ≠ 4 이면 기저 수준 순서대로 번호만 부여
    IDX2G = {k: k + 1 for k in range(final.K)}
    GROUP_LABELS = {g: f"Group {g}" for g in range(1, final.K + 1)}
    GROUP_LABELS_KO = dict(GROUP_LABELS)
    print("⚠️ K ≠ 4: 집단 이름을 직접 지정하고, 8절의 TARGETS 를 다시 정의하세요.")
G2IDX = {g: k for k, g in IDX2G.items()}
GIDS = sorted(G2IDX)
GROUP_NAMES = {g: f"G{g} {GROUP_LABELS[g]}" for g in GIDS}   # 그림·표에 쓰는 이름

# 집단 번호 순서로 재배열한 결과물
E = E_raw[:, [G2IDX[g] for g in GIDS]]                           # (T, K)
pi_est = final.group_proportions()[[G2IDX[g] for g in GIDS]]
post_mat = final.post_[:, [G2IDX[g] for g in GIDS]]

# 사후확률 · 집단 배정
post = pd.DataFrame(post_mat, columns=[f"post_G{g}" for g in GIDS])
traj = agg.loc[use, ["ID"] + agg_cols].reset_index(drop=True).join(post)
traj["group"] = np.array(GIDS)[post_mat.argmax(axis=1)]          # 1–4
traj["group_name"] = traj["group"].map(GROUP_NAMES)
traj["post_max"] = post_mat.max(axis=1)
traj.to_csv(os.path.join(OUT_DIR, "Data_trajectory_group_membership.csv"), index=False, encoding="utf-8-sig")

# ---- 표: 집단 비율 (추정 vs 최빈 배정) · AvePP · OCC (GRoLTS 권장 보고 항목) ----
grp_rows = []
for j, g in enumerate(GIDS):
    sel = traj["group"] == g
    avepp = traj.loc[sel, f"post_G{g}"].mean()
    occ = (avepp / (1 - avepp)) / (pi_est[j] / (1 - pi_est[j]))
    grp_rows.append({"Group": f"G{g}", "Label": GROUP_LABELS[g], "Label_KO": GROUP_LABELS_KO[g],
                     "Estimated %": round(100 * pi_est[j], 1), "Assigned n": int(sel.sum()),
                     "Assigned %": round(100 * sel.mean(), 1), "AvePP": round(avepp, 3),
                     "OCC": round(occ, 1), "n with post ≥ .70": int((sel & (traj["post_max"] >= .70)).sum()),
                     "Polynomial order": final.orders[G2IDX[g]],
                     "Expected W1": round(E[0, j], 2), "Expected W7": round(E[-1, j], 2)})
group_table = pd.DataFrame(grp_rows)
group_table.to_csv(os.path.join(OUT_DIR, "Table_GBTM_groups.csv"), index=False, encoding="utf-8-sig")
print("\n▶ 궤적 집단 요약 (Estimated % = 모형 추정 비율, Assigned % = 최빈 집단 배정 비율)")
display(group_table)

# ---- 모수 표 (집단 번호로 표기) ----
param_tab = final.parameter_table()
param_tab["group"] = param_tab["group"].map(lambda k: f"G{IDX2G[k - 1]}" if k != "" else "")
param_tab = param_tab.sort_values("group", kind="stable")
param_tab.round(4).to_csv(os.path.join(OUT_DIR, "Table_GBTM_parameters.csv"), index=False, encoding="utf-8-sig")
display(param_tab.round(3))

pd.DataFrame(E, index=[f"W{w}" for w in WAVES], columns=[GROUP_NAMES[g] for g in GIDS]).round(2).to_csv(
    os.path.join(OUT_DIR, "Table_GBTM_expected_means.csv"), encoding="utf-8-sig")

# ---- Figure 1: 집단별 궤적 (선 = 모형 추정, 점·오차막대 = 관측 평균 ± 95% CI) ----
GCOL = {1: "#c05621", 2: "#2b6cb0", 3: "#2f855a", 4: "#9b2c2c", 5: "#6b46c1", 6: "#319795"}
fig, ax = plt.subplots(figsize=(7.4, 5.2))
for j, g in enumerate(GIDS):
    sub = traj[traj["group"] == g][agg_cols]
    mu, se = sub.mean(), sub.std() / np.sqrt(sub.notna().sum())
    ax.plot(TIMES, E[:, j], color=GCOL[g], lw=2.4, label=f"G{g} {GROUP_LABELS[g]} ({100 * pi_est[j]:.1f}%)")
    ax.errorbar(TIMES, mu, yerr=1.96 * se, fmt="o", ms=4, color=GCOL[g], alpha=.7, capsize=2)
ax.axhline(np.nanmean(Y_traj[use]), color="grey", lw=.8, ls=":")
ax.text(TIMES[-1] + .1, np.nanmean(Y_traj[use]), "sample\nmean", fontsize=7, color="grey", va="center")
ax.set_xticks(TIMES); ax.set_xticklabels([f"W{w}\n({2017 + w})" for w in WAVES])
ax.set_xlabel("Wave (year)"); ax.set_ylabel("Aggression (raw score, 6–24)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8.5, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "Figure1_GBTM_trajectories.png"), dpi=300, bbox_inches="tight"); plt.show()


## 7. 진단 · Table 1 · 비교쌍 효과크기 · 규범적 수준 · 패널 이탈
비교쌍(`COMPARISONS`)은 여기서 정의되고 8–10절에서 그대로 사용됩니다. 7-1은 **기존 방식(개인 내 표준화 + K-means)을 수정된 자료로 재실행**한 결과입니다. 기저 공격성과 '증가' 라벨이 음의 상관이면 기존 결과의 효과 방향 문제는 평균으로의 회귀 때문입니다.

In [ ]:
# ============================================================
# 7. 진단·기술통계
#   7-1) 기존 방식(개인 내 표준화 + K-means) 민감도 분석 — 수정된 자료로 재실행
#        → 기저 공격성과 '증가/감소' 라벨의 상관을 확인 (평균으로의 회귀 여부)
#   7-2) Table 1: 궤적 집단별 특성 (G1–G4)
#   7-3) 비교쌍별 효과크기 (Cohen's d, FDR) — 예측 분석과 같은 비교쌍
#   7-4) '높은 공격성' 비율의 wave별 변화 — G1 증가가 규범적 수준으로의 증가인지 확인
#   7-5) 패널 이탈 분석: 7개 wave 모두 응답 vs 일부 응답
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy import stats

# ---- 7-1) 기존 방식 민감도 분석 ----
cc = agg.dropna(subset=agg_cols).copy()
Zw = cc[agg_cols].sub(cc[agg_cols].mean(axis=1), axis=0).div(cc[agg_cols].std(axis=1, ddof=0), axis=0).dropna()
cc = cc.loc[Zw.index]
km_lab = KMeans(2, n_init=10, random_state=RANDOM_STATE).fit_predict(Zw)
if Zw[agg_cols[-1]][km_lab == 1].mean() < Zw[agg_cols[-1]][km_lab == 0].mean():
    km_lab = 1 - km_lab                                    # 1 = 상대적 증가
cc["km_increasing"] = km_lab
chk = cc.merge(pred, on="ID")
print(f"▶ 기존 방식 재현 (완전응답 N = {len(cc)}, silhouette k=2 = {silhouette_score(Zw, km_lab):.3f})")
print("  집단별 원점수 평균 — 감소:", cc[cc.km_increasing == 0][agg_cols].mean().round(1).tolist())
print("                      증가:", cc[cc.km_increasing == 1][agg_cols].mean().round(1).tolist())
rtm = pd.DataFrame({v: {"r_with_increasing_label": chk[v].corr(chk["km_increasing"]),
                        "r_with_baseline_aggression": chk[v].corr(chk["Agg_w1"])}
                    for v in ["Agg_w1", "somatic", "attention", "depressive", "parenting_neg", "grit", "self_esteem"]}).T.round(3)
print("  → 기저 공격성(Agg_w1)이 '증가' 라벨과 음의 상관이면, 라벨이 기저 수준을 반대로 반영(평균으로의 회귀)")
display(rtm)
rtm.to_csv(os.path.join(OUT_DIR, "check_regression_to_mean_kmeans.csv"), encoding="utf-8-sig")

# ---- 비교쌍 정의 (8절 예측 분석에서도 그대로 사용) ----
#   (양성 집단, 음성 집단, 설명)
COMPARISONS = {
    "G4_vs_rest": ([4], [1, 2, 3], "Main: G4 high rising-then-declining vs all other groups"),
    "G4_vs_G3":   ([4], [3],       "Elevated start: G4 (stays high) vs G3 (declines)"),
    "G1_vs_G2":   ([1], [2],       "Low start: G1 (increases) vs G2 (stays low)"),
    "G1_vs_G3":   ([1], [3],       "Supplementary: G1 vs G3"),
}
MAIN_COMPARISON = "G4_vs_rest"
if final.K != 4:
    raise ValueError("COMPARISONS 는 4집단 기준입니다. K_FINAL ≠ 4 이면 비교쌍을 다시 정의하세요.")

def fmt_p(p):
    return "" if pd.isna(p) else ("<.001" if p < .001 else f"{p:.3f}".lstrip("0"))

def bh_fdr(p):
    """Benjamini–Hochberg 보정 p"""
    p = np.asarray(p, float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    if ok.sum() == 0:
        return out
    q = p[ok]; n = len(q); o = np.argsort(q)
    adj = np.minimum.accumulate((q[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    tmp = np.empty(n); tmp[o] = np.clip(adj, 0, 1); out[ok] = tmp
    return out

# ---- 7-2) Table 1 ----
dat = traj[["ID", "group", "group_name", "post_max"] + agg_cols].merge(pred, on="ID", how="left")
groups = [GROUP_NAMES[g] for g in GIDS]
rows = []
def add(label, vals, p=""):
    rows.append({"Characteristic": label, **vals, "P": p})

add("N (assigned)", {"Total": f"{len(dat):,}",
                     **{g: f"{(dat.group_name == g).sum():,} ({100 * (dat.group_name == g).mean():.1f}%)" for g in groups}})
add("Estimated proportion, %", {"Total": "100", **{GROUP_NAMES[g]: f"{100 * pi_est[j]:.1f}" for j, g in enumerate(GIDS)}})
cont_vars = ["age"] + agg_cols + [v for v in SCALES] + [v for v in ["income", "father_edu", "mother_edu"] if v in dat]
for v in cont_vars:
    samples = [dat.loc[dat.group_name == g, v].dropna() for g in groups]
    p = stats.f_oneway(*samples).pvalue if all(len(s) > 1 for s in samples) else np.nan
    add(LABELS.get(v, v.replace("Agg_w", "Aggression, wave ")) + ", mean (SD)",
        {"Total": f"{dat[v].mean():.2f} ({dat[v].std():.2f})",
         **{g: f"{s.mean():.2f} ({s.std():.2f})" for g, s in zip(groups, samples)}}, p)
cat_vars = {"female": {0: "Male", 1: "Female"},
            "region_size": {1: "Metropolitan area", 2: "Small- or medium-sized city", 3: "Rural township"},
            "no_smartphone": {0: "Has smartphone", 1: "No smartphone"}}
if "family_type" in dat:
    cat_vars["family_type"] = None
for v, lab in cat_vars.items():
    ct = pd.crosstab(dat[v], dat["group_name"])[groups]
    p = stats.chi2_contingency(ct, correction=False)[1]
    add(LABELS.get(v, v) + ", n (%)", {"Total": "", **{g: "" for g in groups}}, p)
    for lev in ct.index:
        nm = lab.get(int(lev), str(lev)) if lab else str(lev)
        add("   " + nm, {"Total": f"{ct.loc[lev].sum():,} ({100 * ct.loc[lev].sum() / ct.values.sum():.1f})",
                         **{g: f"{ct.loc[lev, g]:,} ({100 * ct.loc[lev, g] / ct[g].sum():.1f})" for g in groups}})
table1 = pd.DataFrame(rows)
_p = pd.to_numeric(table1["P"].replace("", np.nan), errors="coerce")
table1["P_FDR"] = [fmt_p(q) for q in bh_fdr(_p.to_numpy())]
table1["P"] = [fmt_p(q) for q in _p]
table1.to_csv(os.path.join(OUT_DIR, "Table1_by_trajectory_group.csv"), index=False, encoding="utf-8-sig")
print("\n▶ Table 1 (ANOVA / χ²; P_FDR = Benjamini–Hochberg)")
display(table1)

# ---- 7-3) 비교쌍별 효과크기 ----
eff_vars = ["Agg_w1"] + list(SCALES) + ["female", "age"] + [v for v in ["income", "father_edu", "mother_edu"] if v in dat]
eff_rows = []
for cname, (pos, neg, desc) in COMPARISONS.items():
    a_all, b_all = dat[dat["group"].isin(pos)], dat[dat["group"].isin(neg)]
    block = []
    for v in eff_vars:
        a, b = a_all[v].dropna(), b_all[v].dropna()
        sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
        block.append({"comparison": cname, "variable": v, "label": LABELS.get(v, v),
                      "n_pos": len(a), "n_neg": len(b), "mean_pos": a.mean(), "mean_neg": b.mean(),
                      "cohens_d": (a.mean() - b.mean()) / sd if sd > 0 else np.nan,
                      "p": stats.ttest_ind(a, b, equal_var=False).pvalue})
    q = bh_fdr([r["p"] for r in block])
    for r, qq in zip(block, q):
        r["p_FDR"] = qq
    eff_rows += block
effects = pd.DataFrame(eff_rows)
effects.round(4).to_csv(os.path.join(OUT_DIR, "Table1b_pairwise_effect_sizes.csv"), index=False, encoding="utf-8-sig")
eff_wide = effects.pivot(index="label", columns="comparison", values="cohens_d")[list(COMPARISONS)]
eff_wide = eff_wide.reindex([LABELS.get(v, v) for v in eff_vars])
print("\n▶ 비교쌍별 Cohen's d (양수 = 양성 집단이 더 높음). |d| < .20 이면 차이가 작음")
display(eff_wide.round(2))

fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(eff_wide) + 1.2))
im = ax.imshow(eff_wide.to_numpy(float), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(eff_wide.shape[1])); ax.set_xticklabels([c.replace("_", " ") for c in eff_wide.columns], fontsize=8)
ax.set_yticks(range(eff_wide.shape[0])); ax.set_yticklabels(eff_wide.index, fontsize=7.5)
for i in range(eff_wide.shape[0]):
    for j in range(eff_wide.shape[1]):
        v = eff_wide.iat[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if abs(v) > .6 else "black")
plt.colorbar(im, ax=ax, fraction=.03, label="Cohen's d")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "FigS_pairwise_effect_sizes.png"), dpi=300); plt.show()

# ---- 7-4) '높은 공격성' 비율 (기준: wave 1 평균 + 1 SD, 모든 wave 동일 기준) ----
CUT = agg["Agg_w1"].mean() + agg["Agg_w1"].std()
elev_rows = []
for g in GIDS:
    sub = dat[dat["group"] == g]
    row = {"Group": GROUP_NAMES[g]}
    for w, c in zip(WAVES, agg_cols):
        x = sub[c].dropna()
        row[f"W{w} mean"] = round(x.mean(), 2)
        row[f"W{w} % ≥ cut"] = round(100 * (x >= CUT).mean(), 1)
    elev_rows.append(row)
row = {"Group": "Total"}
for w, c in zip(WAVES, agg_cols):
    x = dat[c].dropna(); row[f"W{w} mean"] = round(x.mean(), 2); row[f"W{w} % ≥ cut"] = round(100 * (x >= CUT).mean(), 1)
elev_rows.append(row)
elevated = pd.DataFrame(elev_rows)
elevated.to_csv(os.path.join(OUT_DIR, "TableS_elevated_aggression_by_wave.csv"), index=False, encoding="utf-8-sig")
print(f"\n▶ 높은 공격성 기준 = W1 평균 + 1SD = {CUT:.1f}점. G1의 후기 wave 평균이 전체 평균과 비슷하면 '규범적 수준으로의 증가'")
display(elevated[["Group"] + [c for c in elevated.columns if "%" in c]])

g1_w7 = dat.loc[dat["group"] == 1, agg_cols[-1]].dropna()
tot_w7 = dat[agg_cols[-1]].dropna()
print(f"  G1 W7 평균 {g1_w7.mean():.2f} vs 전체 W7 평균 {tot_w7.mean():.2f} "
      f"(d = {(g1_w7.mean() - tot_w7.mean()) / tot_w7.std():.2f}); "
      f"G1 중 W7 기준 이상 {100 * (g1_w7 >= CUT).mean():.1f}% vs G4 {100 * (dat.loc[dat.group == 4, agg_cols[-1]].dropna() >= CUT).mean():.1f}%")

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for g in GIDS:
    ax.plot(WAVES, [elevated.loc[elevated.Group == GROUP_NAMES[g], f"W{w} % ≥ cut"].item() for w in WAVES],
            "o-", color=GCOL[g], label=GROUP_NAMES[g])
ax.plot(WAVES, [elevated.loc[elevated.Group == "Total", f"W{w} % ≥ cut"].item() for w in WAVES], "k:", label="Total")
ax.set_xlabel("Wave"); ax.set_ylabel(f"% with aggression ≥ {CUT:.1f}")
ax.set_xticks(WAVES); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "FigS_elevated_aggression_by_wave.png"), dpi=300); plt.show()

# ---- 7-5) 패널 이탈 분석 ----
att = agg[["ID", "n_waves_observed", "Agg_w1"]].merge(pred, on="ID")
att["complete"] = (att["n_waves_observed"] == len(WAVES)).astype(int)
attr_rows = []
for v in ["Agg_w1"] + list(SCALES) + ["female"]:
    a, b = att.loc[att["complete"] == 1, v].dropna(), att.loc[att["complete"] == 0, v].dropna()
    sd = np.sqrt((a.var() + b.var()) / 2)
    attr_rows.append({"variable": LABELS.get(v, v), "complete_mean": a.mean(), "incomplete_mean": b.mean(),
                      "SMD": (a.mean() - b.mean()) / sd if sd > 0 else np.nan,
                      "p": stats.ttest_ind(a, b, equal_var=False).pvalue})
attrition = pd.DataFrame(attr_rows).round(3)
print(f"\n▶ 패널 이탈: 7개 wave 모두 응답 {att.complete.sum()}명 vs 일부 응답 {(1 - att.complete).sum()}명 (|SMD| ≥ .10 주목)")
display(attrition[attrition["SMD"].abs() >= .10] if (attrition["SMD"].abs() >= .10).any() else attrition.head())
attrition.to_csv(os.path.join(OUT_DIR, "TableS_attrition.csv"), index=False, encoding="utf-8-sig")


## 8. 궤적 집단 예측 — 반복 nested CV · Table 2
전처리·변수선택·튜닝이 모두 바깥 fold 학습자료 안에서만 이뤄지므로 성능이 낙관적으로 추정되지 않습니다. 성능은 반복 평균 out-of-fold 예측확률 기준입니다.

※ `class_weight`가 선택되면 예측확률이 실제 유병률과 달라집니다(보정곡선 참조). AUC에는 영향이 없지만 확률을 위험도로 해석하지 마세요.

In [ ]:
# ============================================================
# 8. 궤적 집단 예측 — 반복 nested 교차검증 (Random Forest)
#    비교쌍(7절 COMPARISONS):
#      주 분석  G4_vs_rest : 높은 수준 상승 후 감소형(G4) vs 나머지
#               G4_vs_G3   : 높게 출발한 아동 중 높게 유지(G4) vs 감소(G3)
#      G1 분석  G1_vs_G2   : 낮게 출발한 아동 중 증가(G1) vs 낮게 유지(G2)
#               G1_vs_G3   : 보조 비교
#    모형 A = 기저 공격성 제외, 모형 B = 기저 공격성(W1) 포함
#    검증: 바깥 5-fold × N_REPEATS 반복, 안쪽 3-fold RandomizedSearch
#          (전처리·변수선택·RF 튜닝은 모두 바깥 fold의 학습자료 안에서만 수행)
#    성능: 반복 평균 out-of-fold(OOF) 예측확률로 AUC(부트스트랩 95% CI), PR-AUC, Brier
#          + fold별 AUC 평균(SD). 민감도·특이도(Youden)는 OOF에서 정한 값이라 기술적 참고용.
#    SHAP: 첫 반복의 각 바깥 fold 모형으로 해당 test fold를 설명 → 전체 표본의 OOF SHAP
# ============================================================
import shap
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, confusion_matrix
from sklearn.calibration import calibration_curve
from scipy.stats import pointbiserialr, spearmanr

FAST_MODE = bool(int(os.environ.get("KCYPS_FAST", "0")))   # True(1): 빠른 점검용 (반복 1회, 탐색 4회)
N_REPEATS = 1 if FAST_MODE else 3     # 바깥 CV 반복 수 (투고용 결과는 5 이상 권장)
N_OUTER, N_INNER = 5, 3
N_ITER = 4 if FAST_MODE else 8        # 안쪽 RandomizedSearch 후보 수
N_BOOT = 1000
RUN_COMPARISONS = list(COMPARISONS)   # 시간이 부족하면 ["G4_vs_rest", "G1_vs_G2"]

USE_PROVINCE = False     # 17개 시도(원-핫 17열)는 작은 비교쌍(G1 vs G2)에서 잡음으로 상위에 오르기 쉬워 기본 제외
BASE_FEATURES = list(SCALES) + ["female", "age", "region_size", "no_smartphone"] \
    + (["school_region"] if USE_PROVINCE else []) \
    + [c for c in ["income", "father_edu", "mother_edu", "father_job", "mother_job", "family_type"] if c in pred]
CAT_FEATURES = [c for c in ["region_size", "school_region", "father_job", "mother_job", "family_type"] if c in BASE_FEATURES]
MODELS = {"A": BASE_FEATURES, "B": BASE_FEATURES + ["Agg_w1"]}
MODEL_DESC = {"A": "without baseline aggression", "B": "with baseline aggression"}

def make_pipeline(features):
    num_cols = [f for f in features if f not in CAT_FEATURES]
    cat_cols = [f for f in features if f in CAT_FEATURES]
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])
    sel = SelectFromModel(RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample",
                                                 min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1))
    clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([("pre", pre), ("sel", sel), ("clf", clf)])

PARAM_DIST = {
    "sel__threshold": ["median", "mean", -np.inf],                # -inf = 변수선택 안 함
    "clf__max_depth": [None, 6, 12],
    "clf__min_samples_leaf": [5, 10, 20, 40],
    "clf__max_features": ["sqrt", 0.3],
    "clf__class_weight": ["balanced_subsample", None],
}

def to_original(fname, features):
    name = fname.split("__", 1)[-1]
    if name in features:
        return name
    matches = [f for f in features if name.startswith(f + "_")]
    return max(matches, key=len) if matches else name

def run_repeated_cv(X, y, feats, weights=None, n_repeats=N_REPEATS, store_shap=False, tag=""):
    n = len(y)
    oof = np.full((n_repeats, n), np.nan)
    fold_auc, best_params, n_sel = [], [], []
    sv = pd.DataFrame(0.0, index=range(n), columns=feats) if store_shap else None
    base = np.full(n, np.nan)
    for r in range(n_repeats):
        outer = StratifiedKFold(N_OUTER, shuffle=True, random_state=RANDOM_STATE + r)
        for f, (tr, te) in enumerate(outer.split(X, y)):
            inner = StratifiedKFold(N_INNER, shuffle=True, random_state=RANDOM_STATE + 100 * r + f)
            search = RandomizedSearchCV(make_pipeline(feats), PARAM_DIST, n_iter=N_ITER, scoring="roc_auc",
                                        cv=inner, n_jobs=-1, random_state=RANDOM_STATE + r, refit=True)
            fit_kw = {"clf__sample_weight": weights[tr]} if weights is not None else {}
            search.fit(X.iloc[tr], y[tr], **fit_kw)
            best = search.best_estimator_
            p = best.predict_proba(X.iloc[te])[:, 1]
            oof[r, te] = p
            fold_auc.append(roc_auc_score(y[te], p))
            best_params.append(search.best_params_)
            n_sel.append(int(best.named_steps["sel"].get_support().sum()))
            if store_shap and r == 0:
                names = best.named_steps["pre"].get_feature_names_out()[best.named_steps["sel"].get_support()]
                Xte_t = best[:-1].transform(X.iloc[te])
                expl = shap.TreeExplainer(best.named_steps["clf"])          # tree_path_dependent, 확률 단위
                vals = expl.shap_values(Xte_t, check_additivity=False)
                vals = vals[1] if isinstance(vals, list) else (vals[..., 1] if np.ndim(vals) == 3 else vals)
                ev = np.atleast_1d(expl.expected_value)
                base[te] = ev[1] if len(ev) > 1 else ev[0]
                orig = [to_original(nm, feats) for nm in names]
                fold_sv = pd.DataFrame(vals, columns=orig).T.groupby(level=0).sum().T
                sv.loc[te, fold_sv.columns] = fold_sv.to_numpy()
        print(f"    {tag} repeat {r + 1}/{n_repeats}: pooled OOF AUC = {roc_auc_score(y, oof[r]):.3f}")
    return {"oof": oof, "p": np.nanmean(oof, axis=0), "fold_auc": np.array(fold_auc),
            "params": best_params, "n_selected": n_sel, "shap": sv, "base": base}

def boot_ci(y, p, fn=roc_auc_score, n=N_BOOT, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            vals.append(fn(y[i], p[i]))
    return np.percentile(vals, [2.5, 97.5])

def summarize(y, R):
    p = R["p"]
    lo, hi = boot_ci(y, p)
    rep_auc = [roc_auc_score(y, R["oof"][r]) for r in range(R["oof"].shape[0])]
    fpr, tpr, thr = roc_curve(y, p); t = thr[np.argmax(tpr - fpr)]
    tn, fp, fn, tp = confusion_matrix(y, (p >= t).astype(int), labels=[0, 1]).ravel()
    return {"N": len(y), "n_pos": int(y.sum()), "prevalence": y.mean(),
            "AUC": roc_auc_score(y, p), "AUC_95CI": f"{lo:.3f}–{hi:.3f}",
            "AUC_repeat_mean": np.mean(rep_auc), "AUC_repeat_SD": np.std(rep_auc, ddof=1) if len(rep_auc) > 1 else np.nan,
            "AUC_fold_mean": R["fold_auc"].mean(), "AUC_fold_SD": R["fold_auc"].std(ddof=1),
            "PR_AUC": average_precision_score(y, p), "Brier": brier_score_loss(y, p),
            "sens_Youden": tp / (tp + fn), "spec_Youden": tn / (tn + fp),
            "median_n_selected_features": float(np.median(R["n_selected"]))}

model_dat = (traj[["ID", "group", "post_max"]].merge(pred, on="ID")
             .merge(agg[["ID", "Agg_w1"]], on="ID"))
RESULTS, perf_rows = {}, []
t_all = time.time()
for cname in RUN_COMPARISONS:
    pos, neg, desc = COMPARISONS[cname]
    dd = model_dat[model_dat["group"].isin(pos + neg)].reset_index(drop=True)
    y = dd["group"].isin(pos).astype(int).to_numpy()
    print("\n" + "=" * 92 + f"\n■ {cname}: {desc}\n  N = {len(y)}, 양성 {y.sum()} ({100 * y.mean():.1f}%)")
    RESULTS[cname] = {"data": dd, "y": y}
    for m, feats in MODELS.items():
        t0 = time.time()
        R = run_repeated_cv(dd[feats], y, feats, store_shap=True, tag=f"[{cname} / model {m}]")
        S = summarize(y, R)
        RESULTS[cname][m] = {**R, "features": feats, "metrics": S}
        perf_rows.append({"comparison": cname, "description": desc, "model": f"{m} ({MODEL_DESC[m]})", **S,
                          "minutes": round((time.time() - t0) / 60, 1)})
        print(f"  ▶ model {m}: AUC {S['AUC']:.3f} ({S['AUC_95CI']}) | fold AUC {S['AUC_fold_mean']:.3f} "
              f"(SD {S['AUC_fold_SD']:.3f}) | PR-AUC {S['PR_AUC']:.3f} (기준 {S['prevalence']:.3f})")
    # ΔAUC (B − A), 대응 부트스트랩
    pa, pb = RESULTS[cname]["A"]["p"], RESULTS[cname]["B"]["p"]
    rng = np.random.default_rng(RANDOM_STATE); diffs = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            diffs.append(roc_auc_score(y[i], pb[i]) - roc_auc_score(y[i], pa[i]))
    d_obs = roc_auc_score(y, pb) - roc_auc_score(y, pa)
    RESULTS[cname]["delta_auc"] = (d_obs, *np.percentile(diffs, [2.5, 97.5]))
    print(f"  ΔAUC (B − A) = {d_obs:.3f} (95% CI {np.percentile(diffs, 2.5):.3f} to {np.percentile(diffs, 97.5):.3f})")
print(f"\n전체 소요: {(time.time() - t_all) / 60:.1f}분")

perf = pd.DataFrame(perf_rows)
perf["delta_AUC_B_minus_A"] = [
    (f"{RESULTS[c]['delta_auc'][0]:.3f} ({RESULTS[c]['delta_auc'][1]:.3f} to {RESULTS[c]['delta_auc'][2]:.3f})"
     if mdl.startswith("B") else "") for c, mdl in zip(perf["comparison"], perf["model"])]
perf.round(3).to_csv(os.path.join(OUT_DIR, "Table2_prediction_performance.csv"), index=False, encoding="utf-8-sig")
display(perf.round(3))

# 선택된 하이퍼파라미터 빈도 (보충자료)
hp_rows = []
for c in RUN_COMPARISONS:
    for m in MODELS:
        for k, v in pd.DataFrame(RESULTS[c][m]["params"]).astype(str).items():
            for val, cnt in v.value_counts().items():
                hp_rows.append({"comparison": c, "model": m, "parameter": k, "value": val, "n_folds": cnt})
pd.DataFrame(hp_rows).to_csv(os.path.join(OUT_DIR, "TableS_selected_hyperparameters.csv"), index=False, encoding="utf-8-sig")

# ---- ROC · 보정곡선 ----
nC = len(RUN_COMPARISONS)
fig, axes = plt.subplots(2, nC, figsize=(3.6 * nC, 7), squeeze=False)
for j, c in enumerate(RUN_COMPARISONS):
    y = RESULTS[c]["y"]
    for m, ls in [("A", "-"), ("B", "--")]:
        p = RESULTS[c][m]["p"]
        fpr, tpr, _ = roc_curve(y, p)
        axes[0, j].plot(fpr, tpr, ls, label=f"{m} (AUC {roc_auc_score(y, p):.2f})")
        frac, mp = calibration_curve(y, p, n_bins=8, strategy="quantile")
        axes[1, j].plot(mp, frac, ls + "o", ms=3, label=m)
    for a in axes[:, j]:
        a.plot([0, 1], [0, 1], ":", color="grey"); a.legend(frameon=False, fontsize=8)
    axes[0, j].set_title(c.replace("_", " "), fontsize=10)
    axes[0, j].set_xlabel("1 − Specificity"); axes[0, j].set_ylabel("Sensitivity")
    axes[1, j].set_xlabel("Predicted probability"); axes[1, j].set_ylabel("Observed proportion")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "FigS_ROC_calibration.png"), dpi=300); plt.show()


## 9. SHAP 해석 · Figure 2–3 · Table 3
out-of-fold SHAP(각 fold 모형이 학습에 쓰지 않은 아동을 설명). 단위는 예측확률의 변화량, 개별 예측확률 = 기준값 + SHAP 합.

In [ ]:
# ============================================================
# 9. SHAP 해석 (out-of-fold, 첫 반복)
#    - SHAP 단위 = 예측확률 변화량. 원-핫 변수는 원래 변수로 합산
#    - 방향: 변수값과 SHAP의 Spearman ρ (모형 내 방향) + 점이연상관 r (BH-FDR)
#    - Figure 2: G4 vs 나머지(A) · G1 vs G2(B) 전역 중요도 (빨강 = 높을수록 양성 집단 쪽, 파랑 = 반대)
#    - Figure 3: 개별 사례 (G4 아동 1명, G1 아동 1명; 시드 고정 무작위)
#    - 모든 비교쌍·모형의 표/그림은 보충자료로 저장
# ============================================================
TOP_N = 15
FIG2_PANELS = ["G4_vs_rest", "G1_vs_G2"]
POS_COL, NEG_COL, MIX_COL = "#c53030", "#2b6cb0", "#a0aec0"

def shap_table(cname, m):
    R, dd, y = RESULTS[cname][m], RESULTS[cname]["data"], RESULTS[cname]["y"]
    sv = R["shap"]; X = dd[R["features"]]
    imp = sv.abs().mean().sort_values(ascending=False)
    rows = []
    for v in imp.index:
        x = pd.to_numeric(X[v], errors="coerce"); ok = x.notna()
        num_ok = v not in CAT_FEATURES and x[ok].nunique() > 1
        r, p = pointbiserialr(y[ok], x[ok]) if num_ok else (np.nan, np.nan)
        rho_ = spearmanr(x[ok], sv.loc[ok, v]).correlation if num_ok and sv.loc[ok, v].std() > 0 else np.nan
        rows.append({"feature": v, "label": LABELS.get(v, v), "mean_abs_SHAP": imp[v],
                     "rho_value_vs_SHAP": rho_, "r_pb": r, "p": p})
    tab = pd.DataFrame(rows)
    tab["p_FDR"] = bh_fdr(tab["p"].to_numpy())
    tab["direction"] = np.select([tab["rho_value_vs_SHAP"] > .1, tab["rho_value_vs_SHAP"] < -.1],
                                 ["higher → positive group", "higher → comparison group"], "mixed / categorical")
    tab.insert(0, "rank", range(1, len(tab) + 1))
    tab.insert(0, "model", m); tab.insert(0, "comparison", cname)
    return tab

SHAP_TABLES = {(c, m): shap_table(c, m) for c in RUN_COMPARISONS for m in MODELS}
all_shap = pd.concat(SHAP_TABLES.values(), ignore_index=True)
all_shap.round(4).to_csv(os.path.join(OUT_DIR, "Table3_SHAP_all_comparisons.csv"), index=False, encoding="utf-8-sig")
for c in RUN_COMPARISONS:
    print(f"\n■ SHAP — {c} / model A (기준값 평균 {np.nanmean(RESULTS[c]['A']['base']):.3f})")
    display(SHAP_TABLES[(c, "A")].head(TOP_N)[["rank", "label", "mean_abs_SHAP", "direction", "r_pb", "p_FDR"]].round(3))

def bar_panel(ax, tab, title):
    top = tab.head(TOP_N).iloc[::-1]
    cols = [POS_COL if d.startswith("higher → positive") else NEG_COL if d.startswith("higher → comparison") else MIX_COL
            for d in top["direction"]]
    ax.barh(top["label"], top["mean_abs_SHAP"], color=cols)
    for yv, v in enumerate(top["mean_abs_SHAP"]):
        ax.text(v, yv, f" {v:.3f}", va="center", fontsize=7)
    ax.set_title(title, fontsize=10, loc="left"); ax.tick_params(axis="y", labelsize=8)
    ax.set_xlabel("Mean |SHAP value| (Δ predicted probability)", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

from matplotlib.patches import Patch
legend_items = [Patch(color=POS_COL, label="Higher score → target group"),
                Patch(color=NEG_COL, label="Higher score → comparison group"),
                Patch(color=MIX_COL, label="Categorical / mixed")]

# ---- Figure 2 (2개 패널) ----
panels = [c for c in FIG2_PANELS if c in RUN_COMPARISONS]
fig, axes = plt.subplots(1, len(panels), figsize=(6.4 * len(panels), 0.33 * TOP_N + 1.8), squeeze=False)
for j, c in enumerate(panels):
    pos, neg, _ = COMPARISONS[c]
    ttl = f"({chr(65 + j)}) G{'+G'.join(map(str, pos))} vs G{'+G'.join(map(str, neg))}"
    bar_panel(axes[0, j], SHAP_TABLES[(c, "A")], ttl)
fig.legend(handles=legend_items, loc="lower center", ncol=3, frameon=False, fontsize=8)
plt.tight_layout(rect=(0, .05, 1, 1)); plt.savefig(os.path.join(OUT_DIR, "Figure2_SHAP_global.png"), dpi=300); plt.show()

# ---- 보충: 비교쌍 × 모형별 막대·beeswarm ----
for c in RUN_COMPARISONS:
    for m in MODELS:
        fig, ax = plt.subplots(figsize=(6.6, 0.33 * TOP_N + 1.4))
        bar_panel(ax, SHAP_TABLES[(c, m)], f"{c.replace('_', ' ')} — model {m} ({MODEL_DESC[m]})")
        ax.legend(handles=legend_items, frameon=False, fontsize=7, loc="lower right")
        plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"FigS_SHAP_bar_{c}_model{m}.png"), dpi=300)
        plt.close(fig)
    sv, X = RESULTS[c]["A"]["shap"], RESULTS[c]["data"][MODELS["A"]]
    num_top = [f for f in SHAP_TABLES[(c, "A")]["feature"].head(TOP_N) if f not in CAT_FEATURES]
    plt.figure()
    shap.summary_plot(sv[num_top].to_numpy(), X[num_top].astype(float).to_numpy(),
                      feature_names=[LABELS.get(f, f) for f in num_top], show=False, max_display=TOP_N)
    plt.title(c.replace("_", " ") + " — model A", fontsize=10)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"FigS_SHAP_beeswarm_{c}_modelA.png"), dpi=300)
    plt.show() if c in FIG2_PANELS else plt.close()

# ---- 보충: 비교쌍 간 중요도 히트맵 (모형 A) ----
imp_mat = pd.DataFrame({c: SHAP_TABLES[(c, "A")].set_index("label")["mean_abs_SHAP"] for c in RUN_COMPARISONS})
imp_mat = imp_mat.loc[imp_mat.max(axis=1).sort_values(ascending=False).index[:20]]
fig, ax = plt.subplots(figsize=(1.6 * len(RUN_COMPARISONS) + 3.5, 0.32 * len(imp_mat) + 1.2))
im = ax.imshow(imp_mat.to_numpy(), cmap="Blues", aspect="auto")
ax.set_xticks(range(imp_mat.shape[1])); ax.set_xticklabels([c.replace("_", " ") for c in imp_mat.columns], fontsize=8)
ax.set_yticks(range(len(imp_mat))); ax.set_yticklabels(imp_mat.index, fontsize=7.5)
for i in range(imp_mat.shape[0]):
    for j in range(imp_mat.shape[1]):
        ax.text(j, i, f"{imp_mat.iat[i, j]:.3f}", ha="center", va="center", fontsize=6.5)
plt.colorbar(im, ax=ax, fraction=.03, label="Mean |SHAP|")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "FigS_SHAP_importance_heatmap.png"), dpi=300); plt.show()

# ---- Figure 3: 개별 사례 (OOF 설명) ----
def local_case(cname, i, title, fname, max_display=10):
    R, dd = RESULTS[cname]["A"], RESULTS[cname]["data"]
    vals = R["shap"].iloc[i]
    data = pd.to_numeric(dd.loc[i, R["features"]], errors="coerce")
    exp = shap.Explanation(values=vals.to_numpy(), base_values=R["base"][i], data=data.round(2).to_numpy(),
                           feature_names=[LABELS.get(f, f) for f in vals.index])
    plt.figure()
    shap.plots.waterfall(exp, max_display=max_display, show=False)
    plt.title(title, fontsize=10)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, fname), dpi=300, bbox_inches="tight"); plt.show()
    pred_p = R["base"][i] + vals.sum()
    print(f"{title}: 기준값 {R['base'][i]:.3f} + SHAP 합 {vals.sum():+.3f} = {pred_p:.3f} "
          f"(OOF 예측확률 {R['oof'][0, i]:.3f})")
    out = pd.DataFrame({"case": title, "feature": [LABELS.get(f, f) for f in vals.index],
                        "value": data.to_numpy(), "SHAP": vals.to_numpy()})
    return out.reindex(out["SHAP"].abs().sort_values(ascending=False).index).head(max_display)

rng_case = np.random.default_rng(2026)
local_tabs = []
if "G4_vs_rest" in RUN_COMPARISONS:
    y = RESULTS["G4_vs_rest"]["y"]
    i = int(rng_case.choice(np.where(y == 1)[0]))
    local_tabs.append(local_case("G4_vs_rest", i, f"(A) Child in G4 ({GROUP_LABELS[4]})", "Figure3A_SHAP_local_G4.png"))
if "G1_vs_G2" in RUN_COMPARISONS:
    y = RESULTS["G1_vs_G2"]["y"]
    i = int(rng_case.choice(np.where(y == 1)[0]))
    local_tabs.append(local_case("G1_vs_G2", i, f"(B) Child in G1 ({GROUP_LABELS[1]})", "Figure3B_SHAP_local_G1.png"))
if local_tabs:
    local_df = pd.concat(local_tabs, ignore_index=True)
    local_df.round(3).to_csv(os.path.join(OUT_DIR, "TableS_SHAP_local_cases.csv"), index=False, encoding="utf-8-sig")
    display(local_df.round(3))


## 10. 민감도 분석 — 집단 분류 불확실성

In [ ]:
# ============================================================
# 10. 민감도 분석 — 궤적 집단 분류의 불확실성
#    GBTM 집단 배정은 확률적이다 (특히 G1의 AvePP가 가장 낮음).
#    (a) 사후확률 가중: 각 아동의 배정 집단 사후확률을 RF 표본가중치로 사용
#    (b) 확실한 분류만: 사후확률 ≥ .70 인 아동만 분석
#    두 분석 모두 모형 A, 반복 1회 (시간 절약). 주 분석과 AUC·SHAP 순위가 비슷하면 결과가 견고함.
# ============================================================
POST_CUT = 0.70
sens_rows, SENS = [], {}
for cname in RUN_COMPARISONS:
    dd, y = RESULTS[cname]["data"], RESULTS[cname]["y"]
    feats = MODELS["A"]
    # (a) 사후확률 가중
    Ra = run_repeated_cv(dd[feats], y, feats, weights=dd["post_max"].to_numpy(), n_repeats=1,
                         store_shap=True, tag=f"[{cname} / posterior-weighted]")
    # (b) 사후확률 ≥ .70
    keep = (dd["post_max"] >= POST_CUT).to_numpy()
    Rb = run_repeated_cv(dd.loc[keep, feats].reset_index(drop=True), y[keep], feats, n_repeats=1,
                         store_shap=True, tag=f"[{cname} / post ≥ {POST_CUT}]")
    SENS[cname] = {"weighted": Ra, "certain": Rb, "keep": keep}
    for lab, R, yy in [("Main (model A)", RESULTS[cname]["A"], y),
                       ("Posterior-weighted", Ra, y), (f"Posterior ≥ {POST_CUT}", Rb, y[keep])]:
        S = summarize(yy, R)
        sv = R["shap"]
        top5 = ", ".join(LABELS.get(v, v) for v in sv.abs().mean().sort_values(ascending=False).index[:5])
        sens_rows.append({"comparison": cname, "analysis": lab, "N": S["N"], "n_pos": S["n_pos"],
                          "AUC": S["AUC"], "AUC_95CI": S["AUC_95CI"], "PR_AUC": S["PR_AUC"], "top5_SHAP": top5})
sens = pd.DataFrame(sens_rows)
# 주 분석과 SHAP 중요도 순위 일치도 (Spearman ρ)
rho = []
for cname in RUN_COMPARISONS:
    base_imp = RESULTS[cname]["A"]["shap"].abs().mean()
    for lab, key in [("Posterior-weighted", "weighted"), (f"Posterior ≥ {POST_CUT}", "certain")]:
        rho.append(spearmanr(base_imp, SENS[cname][key]["shap"].abs().mean()[base_imp.index]).correlation)
sens["SHAP_rank_rho_vs_main"] = [np.nan if a.startswith("Main") else rho.pop(0) for a in sens["analysis"]]
sens.round(3).to_csv(os.path.join(OUT_DIR, "TableS_sensitivity_classification_uncertainty.csv"),
                     index=False, encoding="utf-8-sig")
print("\n▶ 민감도 분석 (SHAP_rank_rho_vs_main = 주 분석과 변수 중요도 순위의 일치도)")
display(sens.round(3))


## 11. 결과 저장 (모든 그림·표 → zip 다운로드)

In [ ]:
# ============================================================
# 11. 결과 저장 — 모든 표(CSV + 하나의 Excel 통합본)와 그림(PNG)을 zip으로 묶어 다운로드
# ============================================================
meta = {
    "N_panel": int(len(base_ids)), "waves": WAVES, "trajectory_N": int(use.sum()),
    "K_final": int(K_FINAL), "censored_normal": CENSORED,
    "groups": group_table.to_dict(orient="records"),
    "fit": {k: (v.item() if hasattr(v, "item") else v) for k, v in final_stats.items()},
    "comparisons": {c: {"positive": COMPARISONS[c][0], "negative": COMPARISONS[c][1], "desc": COMPARISONS[c][2]}
                    for c in RUN_COMPARISONS},
    "cv": {"repeats": N_REPEATS, "outer_folds": N_OUTER, "inner_folds": N_INNER, "n_iter": N_ITER},
    "delta_auc_B_minus_A": {c: [round(float(x), 3) for x in RESULTS[c]["delta_auc"]] for c in RUN_COMPARISONS},
    "random_state": RANDOM_STATE,
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": __import__("sklearn").__version__, "shap": shap.__version__,
                 "scipy": __import__("scipy").__version__},
}
with open(os.path.join(OUT_DIR, "analysis_summary.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

# 모든 CSV → Excel 통합본 (시트 이름 31자 제한)
csvs = sorted(glob.glob(os.path.join(OUT_DIR, "*.csv")))
xlsx_path = os.path.join(OUT_DIR, "All_tables.xlsx")
with pd.ExcelWriter(xlsx_path) as xw:
    used = set()
    for pth in csvs:
        if os.path.basename(pth).startswith("Data_"):
            continue                                   # 개인별 자료는 Excel 통합본에서 제외
        nm = os.path.splitext(os.path.basename(pth))[0][:31]
        while nm in used:
            nm = nm[:29] + f"_{len(used)}"
        used.add(nm)
        pd.read_csv(pth).to_excel(xw, sheet_name=nm, index=False)

figs = sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".png"))
tabs = sorted(f for f in os.listdir(OUT_DIR) if f.endswith((".csv", ".xlsx", ".json")))
print(f"그림 {len(figs)}개:", figs)
print(f"표·자료 {len(tabs)}개:", tabs)

zip_base = os.path.join(os.path.dirname(os.path.abspath(OUT_DIR)), "KCYPS_results")
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("\nZIP:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
